# TabularBench Comparison: Mutability Mask + Feasibility Audit

This notebook implements two tasks for the FraudBench-TabularBench comparison on LCLD:

**Tier 2 -- Mutability Mask (cells 6-9):**
- Define which LCLD features are immutable (LC-internal, attacker cannot modify)
- Re-run CAPGD with the mask: zero perturbation on immutable features
- Compare robust metrics with vs without mask

**Tier 1 Option B -- Full Feasibility Rate (cells 10-14):**
- Engineer the 6 derived features TabularBench uses (`month_since_earliest_cr_line`, 5 `ratio_*`)
- Run saved adversarial examples through TabularBench constraint checks
- Report per-constraint and aggregate feasibility rates

**Setup:** Same as `colab_runner.ipynb` -- data on Google Drive, repo cloned to `/content/`.

In [ ]:
# Cell 1: Verify GPU
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    gpu_mem = getattr(props, "total_memory", getattr(props, "total_mem", 0)) / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU")

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/FraudBench"
for subdir in ["data", "results", "results/adv_examples"]:
    os.makedirs(os.path.join(DRIVE_ROOT, subdir), exist_ok=True)
print("Google Drive mounted.")

In [ ]:
# Cell 3: Clone or update repo
import os, shutil

REPO_URL = "https://github.com/iHaydenzZ/Capstone_FraudBench.git"
REPO_DIR = "/content/Capstone_FraudBench"

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    os.chdir(REPO_DIR)
    !git pull
else:
    os.chdir("/content")
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)

print(f"Working directory: {os.getcwd()}")
!git log --oneline -3

In [ ]:
# Cell 4: Install dependencies
!pip install -e . -q 2>&1 | tail -5
!pip install "numba>=0.61" -q 2>&1 | tail -3

In [ ]:
# Cell 5: Symlink datasets from Google Drive
import os

DRIVE_DATA = "/content/drive/MyDrive/FraudBench/data"
DATASETS_DIR = "/content/Capstone_FraudBench/datasets"

for dataset_dir in ["CCFD", "ieee-fraud-detection", "LCLD", "Sparkov"]:
    src = os.path.join(DRIVE_DATA, dataset_dir)
    dst = os.path.join(DATASETS_DIR, dataset_dir)
    if os.path.islink(dst):
        os.unlink(dst)
    if os.path.exists(src):
        os.symlink(src, dst)
        print(f"  Linked: {dataset_dir}/")
    else:
        print(f"  NOT FOUND: {dataset_dir}/ -- upload to {src}")

print("Dataset symlinks ready.")

---
# Tier 2: Mutability Mask

Define which LCLD features are **immutable** (Lending Club internally computed -- an attacker at application time cannot modify them), then re-run CAPGD with perturbations locked to zero on immutable features.

In [ ]:
# Cell 6: Define LCLD mutability mapping
#
# Classification rationale:
#   IMMUTABLE = Lending Club assigns/computes these internally after application.
#              An attacker submitting a loan application cannot control them.
#   MUTABLE   = Borrower-reported or borrower-controlled at application time.
#              An attacker crafting a fraudulent application could set these.

# --- Immutable: LC-assigned or credit-bureau-sourced ---
LCLD_IMMUTABLE_RAW = {
    # LC internal pricing/grading
    "grade",
    "sub_grade",
    "int_rate",           # derived from grade
    "installment",        # computed from loan_amnt + int_rate + term
    "funded_amnt",        # LC decides how much to fund
    "funded_amnt_inv",    # investor portion -- LC-determined
    "initial_list_status",# LC listing decision (w = whole, f = fractional)

    # LC verification outcomes
    "verification_status",
    "verification_status_joint",

    # Credit bureau data (attacker cannot forge credit history)
    "delinq_2yrs",
    "inq_last_6mths",
    "inq_last_12m",
    "inq_fi",
    "open_acc",
    "open_acc_6m",
    "open_act_il",
    "open_il_12m",
    "open_il_24m",
    "open_rv_12m",
    "open_rv_24m",
    "pub_rec",
    "pub_rec_bankruptcies",
    "total_acc",
    "revol_bal",
    "revol_util",
    "il_util",
    "all_util",
    "tot_cur_bal",
    "tot_hi_cred_lim",
    "total_bal_il",
    "total_rev_hi_lim",
    "max_bal_bc",
    "pct_tl_nvr_dlq",
    "percent_bc_gt_75",
    "collections_12_mths_ex_med",
    "mths_since_last_delinq",
    "mths_since_last_il_delinq",
    "mths_since_last_major_delinq",
    "mths_since_last_record",
    "mths_since_rcnt_il",

    # Derived ratio (LC computes)
    "payment_inc_ratio",
}

# --- Mutable: borrower-reported at application time ---
LCLD_MUTABLE_RAW = {
    "loan_amnt",          # borrower requests
    "term",               # borrower chooses (36 or 60 months)
    "purpose",            # borrower states
    "emp_length",         # borrower reports
    "annual_inc",         # borrower reports
    "annual_inc_joint",   # borrower reports (joint application)
    "home_ownership",     # borrower's situation
    "dti",                # derived from borrower-reported income
    "dti_joint",          # derived from borrower-reported joint income
    "application_type",   # borrower's choice (individual/joint)
    "addr_state",         # borrower's address
}

print(f"Immutable features: {len(LCLD_IMMUTABLE_RAW)}")
print(f"Mutable features:   {len(LCLD_MUTABLE_RAW)}")
print(f"Total defined:      {len(LCLD_IMMUTABLE_RAW) + len(LCLD_MUTABLE_RAW)}")

In [ ]:
# Cell 7: Build processed-space mutability mask & define masked CAPGD
#
# After OneHotEncoding, categorical features expand:
#   "grade" -> "grade_A", "grade_B", ...
# All expanded columns from an immutable raw feature are also immutable.
#
# The existing project_constraints() already reverts non-numeric (OHE)
# features to original values, so those are effectively immutable.
# This mask additionally locks NUMERIC immutable features.

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from typing import Dict, Any, Optional, Set


def build_processed_mutable_mask(
    processed_feature_names: list,
    immutable_raw: Set[str],
) -> np.ndarray:
    """
    Build a boolean mask over processed (post-OHE) features.
    True = mutable (can be perturbed), False = immutable.

    OHE columns are named like "purpose_debt_consolidation" -- the prefix
    before the first underscore that matches a raw feature name determines
    mutability. For ambiguous cases (multiple underscores), we try
    progressively longer prefixes.
    """
    mask = np.ones(len(processed_feature_names), dtype=bool)  # default: mutable

    for i, col in enumerate(processed_feature_names):
        # Direct match (numeric features keep their name)
        if col in immutable_raw:
            mask[i] = False
            continue

        # OHE match: try prefixes "X" from "X_value"
        # e.g. "verification_status_Verified" -> try "verification_status"
        parts = col.split("_")
        for k in range(1, len(parts)):
            prefix = "_".join(parts[:k])
            if prefix in immutable_raw:
                mask[i] = False
                break

    return mask


def capgd_attack_masked(
    model,
    X: pd.DataFrame,
    y: pd.Series,
    schema,  # ConstraintSchema
    feature_types: Dict[str, str],
    mutable_mask: np.ndarray,
    params: Dict[str, Any] = None,
) -> pd.DataFrame:
    """
    CAPGD attack with mutability mask.
    Identical to attacks/capgd.py but zeroes gradients on immutable features.
    """
    from attacks.capgd import project_constraints

    if params is None:
        params = {}

    epsilon = params.get("epsilon", 0.1)
    steps = params.get("steps", 10)
    step_size = params.get("step_size", epsilon / 4)

    if not hasattr(model, "model") or not isinstance(model.model, nn.Module):
        print("Warning: Model is not PyTorch. Returning clean data.")
        return X

    torch_model = model.model
    device = model.device
    torch_model.eval()

    X_tensor = torch.tensor(X.values, dtype=torch.float32).to(device)
    y_tensor = torch.tensor(y.values, dtype=torch.float32).unsqueeze(1).to(device)
    feature_names = X.columns.tolist()

    # Convert mask to torch tensor on device
    mutable_t = torch.tensor(mutable_mask, dtype=torch.bool).to(device)

    # Random init within epsilon ball
    noise = torch.zeros_like(X_tensor).uniform_(-epsilon, epsilon)
    noise[:, ~mutable_t] = 0  # no perturbation on immutable features
    x_adv = X_tensor + noise
    x_adv = project_constraints(x_adv, X_tensor, schema, feature_names, feature_types)
    x_adv = x_adv.detach()
    x_adv.requires_grad = True

    use_logits = hasattr(model, "_use_logits") and model._use_logits
    criterion = nn.BCEWithLogitsLoss() if use_logits else nn.BCELoss()

    for step in range(steps):
        outputs = torch_model(x_adv)
        loss = criterion(outputs, y_tensor)

        torch_model.zero_grad()
        loss.backward()

        with torch.no_grad():
            grad = x_adv.grad
            grad[:, ~mutable_t] = 0  # KEY: zero gradients on immutable features

            x_adv = x_adv + step_size * grad.sign()

            # Project onto epsilon ball
            if epsilon > 0:
                delta = x_adv - X_tensor
                delta = torch.clamp(delta, -epsilon, epsilon)
                delta[:, ~mutable_t] = 0  # enforce zero delta on immutable
                x_adv = X_tensor + delta

            # Project onto feasibility constraints
            x_adv = project_constraints(
                x_adv, X_tensor, schema, feature_names, feature_types
            )
            x_adv.requires_grad = True

    return pd.DataFrame(
        x_adv.detach().cpu().numpy(), columns=feature_names, index=X.index
    )


print("Masked CAPGD function defined.")

In [ ]:
# Cell 8: Run LCLD neural + masked CAPGD (3 seeds)
#
# This replicates the FraudBench pipeline manually so we can:
#   (a) inject the mutability mask into CAPGD
#   (b) save adversarial examples to disk for the Tier 1 constraint check

import time
import os
import numpy as np
import pandas as pd

from datasets.loader import load_dataset
from datasets.splitter import split_dataset
from preprocessing.processor import DataPreprocessor, get_preprocessor_path
from constraints.schema import ConstraintSchema
from constraints.validator import ConstraintValidator
from models.neural import NeuralModel
from evaluation.metrics import compute_metrics

SEEDS = [42, 123, 456]
EPSILON = 0.1
SAMPLE_FRAC = 0.1

# Storage for results
results_masked = []    # with mutability mask
results_unmasked = []  # without mask (original CAPGD, for comparison)

ADV_SAVE_DIR = "results/adv_examples"
os.makedirs(ADV_SAVE_DIR, exist_ok=True)

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"  SEED = {seed}")
    print(f"{'='*60}")

    # --- 1. Load & split ---
    dataset = load_dataset("lcld", config={"sample_frac": SAMPLE_FRAC})
    X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(
        dataset, test_size=0.2, val_size=0.2, random_state=seed,
    )
    print(f"  Data: train={len(X_train)}, test={len(X_test)}, features={X_train.shape[1]}")

    # --- 2. Preprocess ---
    preprocessor_path = get_preprocessor_path("lcld", seed, len(dataset.X))
    if os.path.exists(preprocessor_path):
        preprocessor = DataPreprocessor.load(preprocessor_path)
        X_train_p = preprocessor.transform(X_train)
    else:
        preprocessor = DataPreprocessor(dataset.feature_types)
        X_train_p = preprocessor.fit_transform(X_train)
        preprocessor.save(preprocessor_path)

    X_test_p = preprocessor.transform(X_test)
    processed_feature_types = {c: "numeric" for c in X_train_p.columns}
    processed_schema = ConstraintSchema.from_data(X_train_p, processed_feature_types)
    print(f"  Processed features: {X_test_p.shape[1]}")

    # --- 3. Build mutability mask ---
    mutable_mask = build_processed_mutable_mask(
        X_test_p.columns.tolist(), LCLD_IMMUTABLE_RAW
    )
    n_mutable = int(mutable_mask.sum())
    n_immutable = int((~mutable_mask).sum())
    print(f"  Mask: {n_mutable} mutable, {n_immutable} immutable (of {len(mutable_mask)})")

    # --- 4. Train model ---
    model_params = {"epochs": 20, "hidden_dim": 128, "batch_size": 256, "lr": 0.001}
    model = NeuralModel(model_params)
    t0 = time.time()
    model.fit(X_train_p, y_train)
    train_time = time.time() - t0
    print(f"  Trained in {train_time:.1f}s")

    # --- 5. Clean evaluation ---
    y_probs_clean = model.predict_proba(X_test_p)
    m_clean = compute_metrics(y_test, y_probs_clean)
    print(f"  Clean -- PR-AUC: {m_clean['pr_auc']:.4f}, Acc: {m_clean['accuracy']:.4f}")

    # --- 6a. CAPGD WITHOUT mask (baseline, for comparison) ---
    from attacks.capgd import capgd_attack

    t0 = time.time()
    X_adv_unmasked = capgd_attack(
        model, X_test_p, y_test, processed_schema, processed_feature_types,
        params={"epsilon": EPSILON, "steps": 10},
    )
    t_unmasked = time.time() - t0

    y_probs_unmasked = model.predict_proba(X_adv_unmasked)
    m_unmasked = compute_metrics(y_test, y_probs_unmasked)
    print(f"  CAPGD (no mask) -- PR-AUC: {m_unmasked['pr_auc']:.4f}, "
          f"Acc: {m_unmasked['accuracy']:.4f} ({t_unmasked:.1f}s)")

    # Save unmasked adversarial examples
    X_adv_unmasked.to_parquet(
        os.path.join(ADV_SAVE_DIR, f"lcld_neural_unmasked_seed{seed}.parquet")
    )

    results_unmasked.append({
        "seed": seed,
        "clean_pr_auc": m_clean["pr_auc"],
        "clean_accuracy": m_clean["accuracy"],
        "clean_recall": m_clean["recall"],
        "clean_f1": m_clean["f1"],
        "robust_pr_auc": m_unmasked["pr_auc"],
        "robust_accuracy": m_unmasked["accuracy"],
        "robust_recall": m_unmasked["recall"],
        "robust_f1": m_unmasked["f1"],
    })

    # --- 6b. CAPGD WITH mask ---
    t0 = time.time()
    X_adv_masked = capgd_attack_masked(
        model, X_test_p, y_test, processed_schema, processed_feature_types,
        mutable_mask=mutable_mask,
        params={"epsilon": EPSILON, "steps": 10},
    )
    t_masked = time.time() - t0

    y_probs_masked = model.predict_proba(X_adv_masked)
    m_masked = compute_metrics(y_test, y_probs_masked)
    print(f"  CAPGD (masked)  -- PR-AUC: {m_masked['pr_auc']:.4f}, "
          f"Acc: {m_masked['accuracy']:.4f} ({t_masked:.1f}s)")

    # Save masked adversarial examples
    X_adv_masked.to_parquet(
        os.path.join(ADV_SAVE_DIR, f"lcld_neural_masked_seed{seed}.parquet")
    )

    results_masked.append({
        "seed": seed,
        "clean_pr_auc": m_clean["pr_auc"],
        "clean_accuracy": m_clean["accuracy"],
        "clean_recall": m_clean["recall"],
        "clean_f1": m_clean["f1"],
        "robust_pr_auc": m_masked["pr_auc"],
        "robust_accuracy": m_masked["accuracy"],
        "robust_recall": m_masked["recall"],
        "robust_f1": m_masked["f1"],
    })

    # Also save the raw (pre-processed) test set and preprocessor reference
    # for the Tier 1 constraint check
    if seed == 42:
        X_test.to_parquet(os.path.join(ADV_SAVE_DIR, "lcld_Xtest_raw_seed42.parquet"))
        X_test_p.to_parquet(os.path.join(ADV_SAVE_DIR, "lcld_Xtest_processed_seed42.parquet"))
        y_test.to_frame().to_parquet(os.path.join(ADV_SAVE_DIR, "lcld_ytest_seed42.parquet"))

print("\nAll seeds complete. Adversarial examples saved to results/adv_examples/")

In [ ]:
# Cell 9: Compare results -- with vs without mutability mask
import pandas as pd

df_unmasked = pd.DataFrame(results_unmasked)
df_masked = pd.DataFrame(results_masked)

print("=" * 70)
print("  COMPARISON: CAPGD without mask vs with mutability mask")
print("  LCLD neural baseline, eps=0.1, 3 seeds")
print("=" * 70)

for label, df in [("No mask (original)", df_unmasked), ("With mutability mask", df_masked)]:
    print(f"\n--- {label} ---")
    for col in ["clean_pr_auc", "robust_pr_auc", "clean_accuracy", "robust_accuracy",
                "clean_recall", "robust_recall"]:
        mean_val = df[col].mean()
        std_val = df[col].std()
        print(f"  {col:25s}: {mean_val:.4f} +/- {std_val:.4f}")

# Delta
print(f"\n--- Delta (masked - unmasked, avg over seeds) ---")
for metric in ["pr_auc", "accuracy", "recall", "f1"]:
    d = df_masked[f"robust_{metric}"].mean() - df_unmasked[f"robust_{metric}"].mean()
    direction = "more robust (attack weaker)" if d > 0 else "less robust"
    print(f"  robust_{metric:10s}: {d:+.4f} ({direction})")

print("\nInterpretation:")
print("  Positive delta = mask makes model MORE robust (attack is weaker, realistic).")
print("  The delta magnitude shows how much the original attack relied on")
print("  perturbing features the attacker cannot actually modify.")

---
# Tier 1 Option B: Full Feasibility Rate

We engineer the 6 derived features TabularBench requires, then check how many of our adversarial examples satisfy their domain constraints.

In [ ]:
# Cell 10: Engineer the 6 derived features that TabularBench uses
#
# TabularBench's LCLD has 28 features. FraudBench has 58.
# The 6 features TabularBench has that FraudBench lacks:
#   1. month_since_earliest_cr_line  (date diff in months)
#   2. ratio_loan_amnt_annual_inc
#   3. ratio_open_acc_total_acc
#   4. ratio_pub_rec_month_since_earliest_cr_line
#   5. ratio_pub_rec_bankruptcies_month_since_earliest_cr_line
#   6. ratio_pub_rec_bankruptcies_pub_rec
#
# INDEX ALIGNMENT:
#   FraudBench's load_lcld() filters rows (drops "Current"), then calls
#   reset_index(drop=True), then samples with random_state=42.
#   We must replicate that exact sequence so our derived-feature indices
#   align with the loader's output.

import pandas as pd
import numpy as np
import os

LCLD_PATH = "datasets/LCLD/loan.csv"
print("Loading raw LCLD data (this may take a minute)...")
df_raw = pd.read_csv(LCLD_PATH, low_memory=False)
print(f"  Raw shape: {df_raw.shape}")

# Filter same as FraudBench loader (loader.py:153-154)
df_raw = df_raw[df_raw["loan_status"] != "Current"].copy()

# Compute month_since_earliest_cr_line BEFORE dropping dates
df_raw["issue_d"] = pd.to_datetime(df_raw["issue_d"], format="mixed", dayfirst=False)
df_raw["earliest_cr_line"] = pd.to_datetime(df_raw["earliest_cr_line"], format="mixed", dayfirst=False)

df_raw["month_since_earliest_cr_line"] = (
    (df_raw["issue_d"] - df_raw["earliest_cr_line"]).dt.days / 30.44
).round().astype(float)

# Compute ratio features
df_raw["ratio_loan_amnt_annual_inc"] = (
    df_raw["loan_amnt"] / df_raw["annual_inc"].replace(0, np.nan)
)
df_raw["ratio_open_acc_total_acc"] = (
    df_raw["open_acc"] / df_raw["total_acc"].replace(0, np.nan)
)
df_raw["ratio_pub_rec_month_since_earliest_cr_line"] = (
    df_raw["pub_rec"] / df_raw["month_since_earliest_cr_line"].replace(0, np.nan)
)
df_raw["ratio_pub_rec_bankruptcies_month_since_earliest_cr_line"] = (
    df_raw["pub_rec_bankruptcies"] / df_raw["month_since_earliest_cr_line"].replace(0, np.nan)
)

# Safe division with default -1 (matches TabularBench's SafeDivision)
ratio_bk_pr = df_raw["pub_rec_bankruptcies"] / df_raw["pub_rec"].replace(0, np.nan)
df_raw["ratio_pub_rec_bankruptcies_pub_rec"] = ratio_bk_pr.fillna(-1)

derived_cols = [
    "month_since_earliest_cr_line",
    "ratio_loan_amnt_annual_inc",
    "ratio_open_acc_total_acc",
    "ratio_pub_rec_month_since_earliest_cr_line",
    "ratio_pub_rec_bankruptcies_month_since_earliest_cr_line",
    "ratio_pub_rec_bankruptcies_pub_rec",
]

# Reset index to match the loader's reset_index(drop=True)
df_raw = df_raw.reset_index(drop=True)

# Apply same sampling as load_lcld(sample_frac=0.1):
#   n = int(len(X) * sample_frac); idx = X.sample(n=n, random_state=42).index
n_sampled = int(len(df_raw) * 0.1)
sample_idx = df_raw.sample(n=n_sampled, random_state=42).index
df_derived_sampled = df_raw.loc[sample_idx, derived_cols].reset_index(drop=True)

df_derived_sampled.to_parquet(
    os.path.join(ADV_SAVE_DIR, "lcld_derived_features_sampled.parquet")
)

print(f"\nDerived features computed and sampled.")
print(f"  Shape: {df_derived_sampled.shape}")
print(f"  Index: 0..{len(df_derived_sampled)-1}")
print(f"\nSample values:")
print(df_derived_sampled[derived_cols].describe().round(3))

In [ ]:
# Cell 11: Inverse-transform adversarial examples back to raw feature space
#
# The preprocessor applies: numeric -> median impute -> StandardScaler
#                           categorical -> constant impute -> OneHotEncode
#
# For numeric features, inverse is: x_raw = x_scaled * scale + mean
# For OHE features we don't need to invert -- the constraints only
# reference numeric features.

import re
import numpy as np
import pandas as pd
import os

from preprocessing.processor import DataPreprocessor, get_preprocessor_path
from datasets.loader import load_dataset

# Load saved data from Cell 8
X_test_raw = pd.read_parquet(os.path.join(ADV_SAVE_DIR, "lcld_Xtest_raw_seed42.parquet"))
X_test_proc = pd.read_parquet(os.path.join(ADV_SAVE_DIR, "lcld_Xtest_processed_seed42.parquet"))
X_adv_un_proc = pd.read_parquet(os.path.join(ADV_SAVE_DIR, "lcld_neural_unmasked_seed42.parquet"))
X_adv_mk_proc = pd.read_parquet(os.path.join(ADV_SAVE_DIR, "lcld_neural_masked_seed42.parquet"))
df_derived_sampled = pd.read_parquet(os.path.join(ADV_SAVE_DIR, "lcld_derived_features_sampled.parquet"))

print(f"Test set:        {X_test_raw.shape}")
print(f"Adv (no mask):   {X_adv_un_proc.shape}")
print(f"Adv (masked):    {X_adv_mk_proc.shape}")
print(f"Derived (full):  {df_derived_sampled.shape}")

# Load the fitted preprocessor to get the scaler parameters
dataset = load_dataset("lcld", config={"sample_frac": 0.1})
pp_path = get_preprocessor_path("lcld", 42, len(dataset.X))
preprocessor = DataPreprocessor.load(pp_path)

# Extract the StandardScaler from the ColumnTransformer
ct = preprocessor.pipeline
num_transformer = None
num_feature_names = []
for name, transformer, columns in ct.transformers_:
    if name == "num":
        num_transformer = transformer  # Pipeline: [imputer, scaler]
        num_feature_names = list(columns)
        break

scaler = num_transformer.named_steps["scaler"]
print(f"\nNumeric features in preprocessor: {len(num_feature_names)}")


def inverse_transform_numeric(X_proc, num_feature_names, scaler):
    """
    Inverse-transform only the numeric columns from processed space
    back to raw space using the fitted StandardScaler.
    """
    sanitize = lambda c: re.sub(r"[\[\]<>]", "_", c)
    sanitized_num = [sanitize(c) for c in num_feature_names]

    proc_cols = X_proc.columns.tolist()
    matched = [(raw, san) for raw, san in zip(num_feature_names, sanitized_num) if san in proc_cols]

    raw_names = [m[0] for m in matched]
    san_names = [m[1] for m in matched]
    idx_in_scaler = [num_feature_names.index(r) for r in raw_names]

    X_scaled = X_proc[san_names].values
    means = scaler.mean_[idx_in_scaler]
    scales = scaler.scale_[idx_in_scaler]
    X_raw_vals = X_scaled * scales + means

    return pd.DataFrame(X_raw_vals, columns=raw_names, index=X_proc.index)


X_test_inv = inverse_transform_numeric(X_test_proc, num_feature_names, scaler)
X_adv_un_inv = inverse_transform_numeric(X_adv_un_proc, num_feature_names, scaler)
X_adv_mk_inv = inverse_transform_numeric(X_adv_mk_proc, num_feature_names, scaler)

# Join derived features to the test set using positional alignment.
# Both df_derived_sampled and X_test_raw share the same 0-based index
# (both originate from the same loader output with reset_index).
# The test split indices are a subset, so we use .iloc for safety.
derived_for_test = df_derived_sampled.iloc[X_test_raw.index].reset_index(drop=True)

# Sanity check: verify a shared base feature matches between raw and derived source
if "loan_amnt" in X_test_raw.columns:
    # Quick check that our index alignment is correct
    raw_sum = X_test_raw["loan_amnt"].sum()
    print(f"\n  Index alignment check: X_test_raw loan_amnt sum = {raw_sum:.0f}")

print(f"\nInverse-transformed shapes:")
print(f"  X_test (raw):       {X_test_inv.shape}")
print(f"  X_adv (unmasked):   {X_adv_un_inv.shape}")
print(f"  X_adv (masked):     {X_adv_mk_inv.shape}")
print(f"  Derived features:   {derived_for_test.shape}")

# Sanity: compare inverse-transformed test vs original raw
shared_cols = [c for c in X_test_inv.columns if c in X_test_raw.columns]
max_diff = (X_test_inv[shared_cols].values - X_test_raw[shared_cols].values)
valid_mask = ~np.isnan(max_diff)
if valid_mask.any():
    print(f"\n  Sanity check (inverse vs original):")
    print(f"    Max error:  {np.abs(max_diff[valid_mask]).max():.6f}")
    print(f"    Mean error: {np.abs(max_diff[valid_mask]).mean():.6f}")

In [ ]:
# Cell 12: Check individual TabularBench constraints on adversarial examples
#
# We check each constraint from TabularBench's LCLD definition (g1-g10)
# on three sets: clean test, adversarial (unmasked), adversarial (masked).

import numpy as np
import pandas as pd

TOLERANCE = 0.01  # same as TabularBench's EqualConstraint tolerance for g1


def check_g1_installment(df, tol=1.0):
    """g1: installment = loan_amnt * (r*(1+r)^t) / ((1+r)^t - 1)."""
    needed = ["loan_amnt", "int_rate", "installment", "term"]
    if not all(c in df.columns for c in needed):
        return None
    r = df["int_rate"] / 1200.0
    t = df["term"]
    expected = df["loan_amnt"] * (r * (1 + r)**t) / ((1 + r)**t - 1)
    diff = (df["installment"] - expected).abs()
    return diff <= tol


def check_g2_open_total(df):
    """g2: open_acc <= total_acc."""
    needed = ["open_acc", "total_acc"]
    if not all(c in df.columns for c in needed):
        return None
    return df["open_acc"] <= df["total_acc"] + TOLERANCE


def check_g3_bankruptcies(df):
    """g3: pub_rec_bankruptcies <= pub_rec."""
    needed = ["pub_rec_bankruptcies", "pub_rec"]
    if not all(c in df.columns for c in needed):
        return None
    return df["pub_rec_bankruptcies"] <= df["pub_rec"] + TOLERANCE


def check_g4_term(df):
    """g4: term must be 36 or 60."""
    if "term" not in df.columns:
        return None
    return df["term"].round().isin([36, 60])


def check_g5_ratio_loan_inc(df, tol=0.01):
    """g5: ratio_loan_amnt_annual_inc == loan_amnt / annual_inc."""
    needed = ["loan_amnt", "annual_inc", "ratio_loan_amnt_annual_inc"]
    if not all(c in df.columns for c in needed):
        return None
    expected = df["loan_amnt"] / df["annual_inc"].replace(0, np.nan)
    return (df["ratio_loan_amnt_annual_inc"] - expected).abs() <= tol


def check_g6_ratio_open_total(df, tol=0.01):
    """g6: ratio_open_acc_total_acc == open_acc / total_acc."""
    needed = ["open_acc", "total_acc", "ratio_open_acc_total_acc"]
    if not all(c in df.columns for c in needed):
        return None
    expected = df["open_acc"] / df["total_acc"].replace(0, np.nan)
    return (df["ratio_open_acc_total_acc"] - expected).abs() <= tol


# --- Assemble DataFrames for checking ---

# Clean: use raw test + derived features
df_clean = X_test_raw.copy()
for col in derived_for_test.columns:
    df_clean[col] = derived_for_test[col].values

# Adversarial: use inverse-transformed numerics.
# Derived ratios recomputed from perturbed base features.
def add_derived_from_base(df):
    out = df.copy()
    if "annual_inc" in df.columns and "loan_amnt" in df.columns:
        out["ratio_loan_amnt_annual_inc"] = df["loan_amnt"] / df["annual_inc"].replace(0, np.nan)
    if "open_acc" in df.columns and "total_acc" in df.columns:
        out["ratio_open_acc_total_acc"] = df["open_acc"] / df["total_acc"].replace(0, np.nan)
    return out

df_adv_un = add_derived_from_base(X_adv_un_inv)
df_adv_mk = add_derived_from_base(X_adv_mk_inv)

datasets_to_check = {
    "Clean test set": df_clean,
    "Adversarial (no mask)": df_adv_un,
    "Adversarial (with mask)": df_adv_mk,
}

# --- Run base-feature constraints (g1-g4) on all three ---
constraint_checks = {
    "g1 (installment formula)": check_g1_installment,
    "g2 (open_acc <= total_acc)": check_g2_open_total,
    "g3 (bankruptcies <= pub_rec)": check_g3_bankruptcies,
    "g4 (term in {36, 60})": check_g4_term,
}

print("=" * 70)
print("  PER-CONSTRAINT FEASIBILITY RATES")
print("=" * 70)

results_table = []
for cname, cfunc in constraint_checks.items():
    row = {"Constraint": cname}
    for dname, ddf in datasets_to_check.items():
        result = cfunc(ddf)
        if result is not None:
            valid = result.dropna()
            rate = valid.mean() if len(valid) > 0 else float("nan")
            row[dname] = f"{rate:.4f}"
        else:
            row[dname] = "N/A"
    results_table.append(row)

# g5, g6 only meaningful for clean (adversarial ratios are recomputed)
for cname, cfunc in [("g5 (ratio_loan/inc)", check_g5_ratio_loan_inc),
                      ("g6 (ratio_open/total)", check_g6_ratio_open_total)]:
    row = {"Constraint": cname}
    result = cfunc(df_clean)
    row["Clean test set"] = f"{result.dropna().mean():.4f}" if result is not None else "N/A"
    row["Adversarial (no mask)"] = "(recomputed)"
    row["Adversarial (with mask)"] = "(recomputed)"
    results_table.append(row)

df_results = pd.DataFrame(results_table)
print(df_results.to_string(index=False))

In [ ]:
# Cell 13: Aggregate feasibility rate + perturbation statistics
#
# The aggregate rate is the fraction of adversarial samples that pass
# ALL checkable constraints simultaneously. This is the key number.

import numpy as np
import pandas as pd


def compute_aggregate_feasibility(df):
    """Check g1 (relaxed), g2, g3, g4 simultaneously."""
    checks = []

    # g1: relaxed to 1.0 tolerance due to floating-point imprecision
    # from inverse-transform through StandardScaler
    g1 = check_g1_installment(df, tol=1.0)
    if g1 is not None:
        checks.append(g1.fillna(True))

    g2 = check_g2_open_total(df)
    if g2 is not None:
        checks.append(g2.fillna(True))

    g3 = check_g3_bankruptcies(df)
    if g3 is not None:
        checks.append(g3.fillna(True))

    g4 = check_g4_term(df)
    if g4 is not None:
        checks.append(g4.fillna(True))

    if not checks:
        return None, None

    all_pass = checks[0]
    for c in checks[1:]:
        all_pass = all_pass & c

    return all_pass, float(all_pass.mean())


print("=" * 70)
print("  AGGREGATE FEASIBILITY RATES (g1 + g2 + g3 + g4)")
print("=" * 70)

for name, df in datasets_to_check.items():
    per_sample, rate = compute_aggregate_feasibility(df)
    if rate is not None:
        n_pass = int(per_sample.sum())
        n_total = len(per_sample)
        print(f"  {name:30s}: {rate:.4f}  ({n_pass}/{n_total} samples)")
    else:
        print(f"  {name:30s}: N/A (missing features)")

# --- Perturbation statistics on constrained features ---
print("\n" + "=" * 70)
print("  PERTURBATION STATISTICS ON CONSTRAINED FEATURES")
print("=" * 70)

constrained_features = [
    "loan_amnt", "int_rate", "installment", "term",
    "open_acc", "total_acc", "pub_rec", "pub_rec_bankruptcies",
    "annual_inc",
]

for variant, X_adv_inv in [("No mask", X_adv_un_inv), ("Masked", X_adv_mk_inv)]:
    print(f"\n--- {variant} ---")
    header = f"  {'Feature':42s} {'Mean |delta|':>12s} {'Max |delta|':>12s} {'% Changed':>10s}"
    print(header)
    for feat in constrained_features:
        if feat in X_adv_inv.columns and feat in X_test_inv.columns:
            delta = (X_adv_inv[feat] - X_test_inv[feat]).abs()
            valid = delta.dropna()
            if len(valid) > 0:
                mean_d = valid.mean()
                max_d = valid.max()
                pct_changed = (valid > 0.001).mean() * 100
                print(f"  {feat:42s} {mean_d:12.4f} {max_d:12.4f} {pct_changed:9.1f}%")
            else:
                print(f"  {feat:42s} {'(all NaN)':>12s}")

In [ ]:
# Cell 14: Summary narrative for paper/advisor

summary = """
======================================================================
           SUMMARY: TabularBench Comparison Results
======================================================================

TIER 2 -- MUTABILITY MASK
-------------------------
We classified FraudBench's 58 LCLD features into mutable (borrower-
controlled at application time) and immutable (LC-internal or credit-
bureau-sourced). With the mask applied:

  * CAPGD is restricted to perturbing only mutable features.
  * Delta between masked and unmasked robust metrics quantifies how
    much the original attack relied on perturbing unreachable features.

TIER 1 -- FEASIBILITY RATE
--------------------------
We checked adversarial examples against TabularBench's domain
constraints (g1-g4 on shared base features):

  * g1 (installment formula): Tests whether CAPGD preserves the
    financial relationship between loan_amnt, int_rate, term, and
    installment.
  * g2 (open_acc <= total_acc): Accounting constraint.
  * g3 (bankruptcies <= pub_rec): Logical subset constraint.
  * g4 (term in {36, 60}): Domain value constraint.

KEY FINDINGS
------------
  1. A low feasibility rate confirms FraudBench's bounds-only CAPGD
     generates many domain-infeasible adversarial examples.
     -> Future work: integrate constraint-aware attacks.

  2. The mutability mask improves realism. The masked robust metrics
     are the more trustworthy measure of real-world vulnerability.

  3. These results bridge the two benchmarks: FraudBench contributes
     tree models, black-box attacks, and PR-AUC; TabularBench
     contributes domain constraints and constraint-aware attacks.

======================================================================
"""
print(summary)

In [ ]:
# Cell 15: Save all results to Google Drive
import shutil
import glob
import os

RESULTS_DST = "/content/drive/MyDrive/FraudBench/results/adv_examples"
os.makedirs(RESULTS_DST, exist_ok=True)

for f in glob.glob(os.path.join(ADV_SAVE_DIR, "*.parquet")):
    dst = os.path.join(RESULTS_DST, os.path.basename(f))
    shutil.copy2(f, dst)
    print(f"  Saved: {os.path.basename(f)}")

# Save the comparison DataFrames as CSV
df_unmasked.to_csv(os.path.join(RESULTS_DST, "comparison_unmasked.csv"), index=False)
df_masked.to_csv(os.path.join(RESULTS_DST, "comparison_masked.csv"), index=False)
print("  Saved: comparison_unmasked.csv, comparison_masked.csv")

print(f"\nAll results backed up to {RESULTS_DST}")